# Reusable City Scoring and Prediction Publication

**Purpose.** Score one selected city with the **existing fitted A/B Random Forest** and publish its model and baseline predictions independently of training. This is the reusable scoring workflow extracted from `04_2_city_c_fixes` after the Montreal onboarding exercise.

**No training / retraining occurs in this notebook.** `03_price_model.ipynb` remains the original reference implementation; `04_1` and `04_2` retain the evaluation and fix evidence.

**Safety:** Dry run by default. Publication requires an explicit switch. Replacing predictions for a city that already exists requires a **second** explicit switch. The notebook does not write to the ETL tables or dashboard/Genie definitions.

## 1. Configuration

In [0]:
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from pyspark.sql import functions as F

# Databricks widget: use an existing city slug in the gold feature table.
dbutils.widgets.text("city", "montreal")
CITY = dbutils.widgets.get("city").strip().lower()
assert CITY and CITY.replace("_", "").replace("-", "").isalnum(), "Invalid city slug"

SCHEMA = "airbnb_lakehouse"
FEATURES_TABLE = f"{SCHEMA}.gold_listings_features"
PREDICTIONS_TABLE = f"{SCHEMA}.gold_model_predictions"
ANALYTICS_VIEW = f"{SCHEMA}.gold_listings_analytics"
MODEL_ID = "m-f6812ff886404598946420b8d3ea713a"
MODEL_URI = f"models:/{MODEL_ID}"
TRAIN_CITIES = ("toronto", "vancouver")

# Switches are intentionally OFF by default.
PUBLISH = False
ALLOW_REPLACE_EXISTING_CITY = False

num_features = [
    "accommodates", "bedrooms", "beds", "bathrooms_clean",
    "latitude", "longitude", "hosts_time_as_host_years",
    "minimum_nights", "availability_365"
]
cat_features = ["room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost"]
feature_cols = num_features + cat_features

print("Scoring city:", CITY)
print("Frozen model:", MODEL_URI)
print("Publish:", PUBLISH, "| Allow replacing existing city:", ALLOW_REPLACE_EXISTING_CITY)

## 2. Read existing data and reconstruct the original baseline (read-only)

Mirrors notebook 03 exactly: separate `train_test_split(test_size=0.20, random_state=42)` for each original training city, group training prices by `(room_type, accommodates)`, and use the global A/B training median for unseen combinations. **Never use the target city's prices to fit its baseline.**

The source row order matters to this split, so the baseline check below is a useful reproducibility check, not a guarantee against future input changes.

In [0]:
source = spark.table(FEATURES_TABLE)
source_counts = {r["city"]: r["count"] for r in source.groupBy("city").count().collect()}
print("Available gold city counts:", source_counts)
assert CITY in source_counts and source_counts[CITY] > 0, "City is missing from gold features: run ETL first"
assert all(source_counts.get(c, 0) > 0 for c in TRAIN_CITIES), "Original A/B training cities must be available"

df_ab = source.filter(F.col("city").isin(*TRAIN_CITIES)).toPandas()
train_parts, test_parts = [], []
for city_name in TRAIN_CITIES:
    city_frame = df_ab.loc[df_ab["city"] == city_name].copy()
    tr, te = train_test_split(city_frame, test_size=0.20, random_state=42)
    train_parts.append(tr)
    test_parts.append(te)
df_train_ab = pd.concat(train_parts, ignore_index=True)
df_test_ab = pd.concat(test_parts, ignore_index=True)

baseline_lookup = (
    df_train_ab.groupby(["room_type", "accommodates"], as_index=False)["price_clean"]
    .median()
    .rename(columns={"price_clean": "baseline_price"})
)
global_train_median = float(df_train_ab["price_clean"].median())

def attach_baseline(frame):
    result = frame.merge(baseline_lookup, on=["room_type", "accommodates"], how="left", validate="many_to_one")
    result["baseline_price"] = result["baseline_price"].fillna(global_train_median)
    return result

check = attach_baseline(df_test_ab)
ab_baseline_mae = mean_absolute_error(check["price_clean"], check["baseline_price"])
print(f"Reconstructed held-out A/B baseline MAE: {ab_baseline_mae:.2f} CAD; notebook 03 reported 109.32 CAD")
print("Global training median:", global_train_median)
# Strict check protects baseline comparability; review the source data/order if it fails.
assert abs(ab_baseline_mae - 109.32) < 0.05, "A/B baseline no longer matches notebook 03: investigate before publishing"

## 3. Load the frozen fitted model and score the selected city (read-only)

The fitted scikit-learn pipeline includes preprocessing. The model predicts log-prices; apply `expm1` to return to CAD. Scoring **does not modify** the model or train it on the target city.

In [0]:
df_city = source.filter(F.col("city") == CITY).toPandas()
required = set(feature_cols + ["id", "city", "price_clean"])
assert required.issubset(df_city.columns), f"Missing required columns: {required - set(df_city.columns)}"
assert len(df_city) == source_counts[CITY], "Row count mismatch"
assert not df_city[["id", "city"]].duplicated().any(), "Duplicate listing keys: investigate"
assert not df_city[feature_cols + ["price_clean"]].isna().any().any(), "Null feature/label: document and investigate"
assert np.isfinite(df_city["price_clean"].to_numpy(dtype=float)).all(), "Invalid actual prices"

model_pipeline = mlflow.sklearn.load_model(MODEL_URI)
assert {"preprocessor", "regressor"}.issubset(model_pipeline.named_steps), "Unexpected MLflow model pipeline"
# Confirm model's fitted columns match the expected notebook 03 features.
preprocessor = model_pipeline.named_steps["preprocessor"]
if hasattr(preprocessor, "feature_names_in_"):
    assert set(preprocessor.feature_names_in_) == set(feature_cols), "Model feature list differs from notebook 03"

predicted_log_price = model_pipeline.predict(df_city[feature_cols])
predicted_price = np.expm1(predicted_log_price)
assert len(predicted_price) == len(df_city) and np.isfinite(predicted_price).all(), "Invalid model predictions"

results = attach_baseline(df_city)
results["predicted_price"] = predicted_price
results["absolute_error"] = (results["price_clean"] - results["predicted_price"]).abs()
model_mae = mean_absolute_error(results["price_clean"], results["predicted_price"])
baseline_mae = mean_absolute_error(results["price_clean"], results["baseline_price"])
print(f"Scored {len(results):,} {CITY} listings")
print(f"Model MAE: {model_mae:.2f} CAD | A/B baseline MAE: {baseline_mae:.2f} CAD")
print(f"Improvement vs baseline: {(1 - model_mae / baseline_mae) * 100:.1f}%")
if CITY == "montreal":
    assert abs(model_mae - 79.68) < 0.05, "Original Montreal score changed; investigate before publishing"
display(results[["id", "city", "room_type", "accommodates", "price_clean", "baseline_price", "predicted_price", "absolute_error"]].head(10))

## 4. Prepare publication rows and inspect existing table (read-only)

Mirror notebook 03's output schema and two-decimal rounding. We cast to the existing Delta schema rather than recreating the prediction table. Since `04_2` has already published Montreal, a new execution for Montreal should **remain a dry run** unless you deliberately opt into replacing its current rows.

In [0]:
pred_before = spark.table(PREDICTIONS_TABLE)
counts_before = {r["city"]: r["count"] for r in pred_before.groupBy("city").count().collect()}
print("Prediction rows before:", counts_before)
expected_output = {"id", "city", "room_type", "accommodates", "actual_price", "baseline_price", "predicted_price", "absolute_error"}
assert set(pred_before.columns) == expected_output, "Unexpected prediction table schema; STOP"

publication_pd = results[["id", "city", "room_type", "accommodates", "price_clean", "baseline_price", "predicted_price"]].copy()
publication_pd.rename(columns={"price_clean": "actual_price"}, inplace=True)
for col in ("actual_price", "baseline_price", "predicted_price"):
    publication_pd[col] = publication_pd[col].astype(float).round(2)
publication_pd["absolute_error"] = (publication_pd["actual_price"] - publication_pd["predicted_price"]).abs().round(2)

publication_spark = spark.createDataFrame(publication_pd)
existing_schema = pred_before.schema
publication_spark = publication_spark.select(*[
    F.col(field.name).cast(field.dataType).alias(field.name) for field in existing_schema.fields
])
assert publication_spark.count() == len(results)
assert publication_spark.filter(F.col("city") != CITY).count() == 0
print(f"Prepared {len(results):,} {CITY} rows; target already has {counts_before.get(CITY, 0):,}")
display(publication_spark.limit(5))

## 5. Optional city-scoped publication (the only write)

**Read before enabling:** `PUBLISH = False` makes this a dry run. To publish a new city, set `PUBLISH = True`. To *replace* a city already present in the table, also set `ALLOW_REPLACE_EXISTING_CITY = True` after checking that this is intentional. This will overwrite **only the selected city's** rows using `replaceWhere`; other cities should be preserved. Do not rerun notebook 03's full-table export.

For a reproducible before/after comparison, capture the output of Section 4 and the Delta version before enabling a write.

In [0]:
if not PUBLISH:
    print("DRY RUN: no tables modified.")
else:
    if counts_before.get(CITY, 0) > 0 and not ALLOW_REPLACE_EXISTING_CITY:
        raise RuntimeError(
            f"{CITY} already has {counts_before[CITY]} prediction rows. "
            "Set ALLOW_REPLACE_EXISTING_CITY=True only if replacement is intentional."
        )
    print("Delta version before publication:")
    display(spark.sql(f"DESCRIBE HISTORY {PREDICTIONS_TABLE} LIMIT 1"))
    (publication_spark.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"city = '{CITY}'")
        .saveAsTable(PREDICTIONS_TABLE))
    print(f"Published {len(results):,} rows for {CITY} using a city-scoped overwrite")

## 6. Read-only downstream validation

In [0]:
counts_after = {r["city"]: r["count"] for r in spark.table(PREDICTIONS_TABLE).groupBy("city").count().collect()}
print("Prediction rows after:", counts_after)
for city_name, n_before in counts_before.items():
    if city_name != CITY:
        assert counts_after.get(city_name, 0) == n_before, f"Count changed unexpectedly for {city_name}"
if PUBLISH:
    assert counts_after.get(CITY) == len(results), "Published city row count mismatch"

if spark.catalog.tableExists(ANALYTICS_VIEW):
    analytics_counts = {r["city"]: r["count"] for r in spark.table(ANALYTICS_VIEW).groupBy("city").count().collect()}
    print("Genie analytics view:", analytics_counts)
    if PUBLISH:
        assert analytics_counts.get(CITY) == len(results), "City missing or duplicated in Genie analytics view"
else:
    print("Analytics view is absent; verify Genie configuration separately")

if not PUBLISH:
    print("No write requested. Counts reflect current published data, not the prepared dry-run rows.")

## 7. Operational notes

- **Generalization:** For a newly onboarded city, run the existing city-parameterized ETL, then run this notebook with its `city` widget set to the new city and `PUBLISH = False` initially. Review results before publishing.
- **Dashboard:** Rerun and republish `05_dashboard.ipynb` after new predictions are available; generated narrative titles are refreshed at notebook execution time.
- **Genie:** Verify counts and generated SQL after publishing; its `gold_listings_analytics` view depends on the prediction table.
- **Evaluation:** The A/B predictions table includes training rows; MAE computed from that entire table is *not* the A/B held-out MAE (68.53 CAD) recorded in notebook 03.
- **Safety:** Validation above confirms counts and feature compatibility, but does not guarantee that the same listing IDs and values are preserved in other cities. For a strict audit, compare keyed row checksums or use Delta version history.